**ADABOOST**

 We will be using the [Indian Liver Patient](https://www.kaggle.com/datasets/uciml/indian-liver-patient-records) dataset. Our task is to predict whether a patient suffers from a liver disease using 10 features including Albumin, age and gender. However, this time, we'll be training an AdaBoost ensemble to perform the classification task. In addition, given that this dataset is imbalanced, we'll be using the ROC AUC score as a metric instead of accuracy.

In [34]:
#Import Libraries
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error as MSE

In [35]:
#Instatiate dt and ada
dt = DecisionTreeClassifier(max_depth=2, random_state=1)
ada = AdaBoostClassifier(estimator=dt, n_estimators=180, random_state=1)

In [36]:
df = pd.read_csv('/content/indian_liver_patient.csv')
df = df.dropna()
# Encode 'Gender' column
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

X = df.drop(columns=['Dataset'])
y = df['Dataset']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state =1)

In [37]:
#Fit ada to the training set
ada.fit(X_train, y_train)

#Compute the probabilities of obtaining the positive class
y_pred_proba = ada.predict_proba(X_test)[:,1]


In [38]:
#Evalute the roc_auc of ada
ada_roc_auc = roc_auc_score(y_test, y_pred_proba)

#Print roc_auc
print('ROC AUC score: {:.2f}'.format(ada_roc_auc))


ROC AUC score: 0.69


**GRADIENT BOOSTING**

We will be using the [Bike Sharing Demand](https://www.kaggle.com/c/bike-sharing-demand) dataset. Recall that your task is to predict the bike rental demand using historical weather data from the Capital Bikeshare program in Washington, D.C.. For this purpose, you'll be using a gradient boosting regressor.

In [39]:
#Instantiate gb
gb = GradientBoostingRegressor(max_depth=4, n_estimators=200, random_state=2)

In [40]:
#Import the dataset
train_df = pd.read_csv('/content/train.csv')
test_df = pd.read_csv('/content/test.csv')

# Preprocess 'datetime' column for both train and test sets
train_df['datetime'] = pd.to_datetime(train_df['datetime'])
test_df['datetime'] = pd.to_datetime(test_df['datetime'])

train_df['year'] = train_df['datetime'].dt.year
train_df['month'] = train_df['datetime'].dt.month
train_df['day'] = train_df['datetime'].dt.day
train_df['hour'] = train_df['datetime'].dt.hour
train_df['dayofweek'] = train_df['datetime'].dt.dayofweek

test_df['year'] = test_df['datetime'].dt.year
test_df['month'] = test_df['datetime'].dt.month
test_df['day'] = test_df['datetime'].dt.day
test_df['hour'] = test_df['datetime'].dt.hour
test_df['dayofweek'] = test_df['datetime'].dt.dayofweek

# Drop the original 'datetime' column
train_df = train_df.drop('datetime', axis=1)
test_df = test_df.drop('datetime', axis=1)

# Drop 'casual' and 'registered' from the training features for consistent features with test_df
X_full_train = train_df.drop(['count', 'casual', 'registered'], axis=1)
y_full_train = train_df['count']

# Split the full training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_full_train, y_full_train, test_size=0.2, random_state=42)

# The X_test will be the competition's test set
X_test = test_df

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of X_test (competition): {X_test.shape}")

Shape of X_train: (8708, 13)
Shape of y_train: (8708,)
Shape of X_val: (2178, 13)
Shape of y_val: (2178,)
Shape of X_test (competition): (6493, 13)


In [41]:
#Fit gb to the training set
gb.fit(X_train, y_train)

GradientBoostingRegressor(max_depth=4, n_estimators=200, random_state=2)

In [42]:
# Predict on the validation set
y_pred_gb_val = gb.predict(X_val)

#Compute MSE for the validation set
mse_val = MSE(y_val, y_pred_gb_val)
rmse = mse_val ** (1/2)

#Print rmse
print('Validation set RMSE of gb: {:.3f}'.format(rmse))

Validation set RMSE of gb: 46.053


**STOCHASTIC GRADIENT BOOSTING**

We will solve this bike count regression problem using stochastic gradient boosting.

In [45]:
#Instantiate sgbr
sgbr = GradientBoostingRegressor(max_depth=4, subsample=0.9, max_features=0.75, n_estimators=200, random_state=2)

In [46]:
#Fit sgbr to training and predict on test
sgbr.fit(X_train, y_train)
y_pred_sgbr = sgbr.predict(X_val)

In [47]:
#Compute test set RMSE
mse = MSE(y_val, y_pred_sgbr)
rmse = mse ** (1/2)

print('Test set RMSE of sgbr: {:.3f}'.format(rmse))

Test set RMSE of sgbr: 45.142


The stochastic gradient boosting regressor achieves a lower test set RMSE than the gradient boosting regressor